# nano-gpt-lab: universal Colab / Kaggle runner

Same notebook, both platforms. Two-tier workflow:
- **Local machine**: first-principles build, unit tests, tiny models, debugging (run this notebook locally too).
- **Colab/Kaggle T4**: larger NanoGPT, TinyStories, OpenWebText subset, FlashAttention benchmarks, GPT-2 ~124M.

The code comes from GitHub automatically (git clone, with a zipball fallback) - no manual zip upload needed:
https://github.com/Th3Samaritan/nano-gpt-lab

**Colab**: outputs mirror to `/content/drive/MyDrive/nano-gpt-lab/`
**Kaggle**: outputs persist automatically in `/kaggle/working/nano-gpt-lab/` (saved to your Kaggle account as the notebook output). On the T4x2 accelerator both GPUs are used automatically (DataParallel splits the batch and averages gradients - same recipe, ~2x speed)

In [ ]:
# --- 0. WHERE ARE WE? ------------------------------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""
print("colab:", IN_COLAB, "| kaggle:", IN_KAGGLE, "| local:", not (IN_COLAB or IN_KAGGLE))

In [ ]:
# --- 1. GET THE REPO (from GitHub - no manual zip needed) -----------------
# Priority: already present locally (git pull keeps it fresh) ->
# git clone from REPO_URL -> zipball download (wget-style) -> manual zip.
import os, sys, zipfile, glob, subprocess, io, urllib.request

REPO_NAME = "nano-gpt-lab"
REPO_URL = "https://github.com/Th3Samaritan/nano-gpt-lab.git"
ZIPBALL = REPO_URL[:-4] + "/archive/refs/heads/main.zip"

def has_repo(d):
    return d and os.path.exists(os.path.join(d, "scripts", "train.py"))

def find_repo_local():
    here = os.path.abspath("")
    for cand in [here, os.path.dirname(here),
                 os.path.join(here, REPO_NAME),
                 os.path.join(os.path.dirname(here), REPO_NAME)]:
        if has_repo(cand):
            return cand
    return None

def extract_zip(zf, dest):
    with zipfile.ZipFile(zf) as z:
        # github zips nest everything under nano-gpt-lab-main/
        z.extractall(dest)
        for root in [os.path.join(dest, "nano-gpt-lab-main"),
                     os.path.join(dest, REPO_NAME)]:
            if has_repo(root):
                return root
    return dest if has_repo(dest) else None

repo = find_repo_local()

if repo and os.path.isdir(os.path.join(repo, ".git")):
    # Already have a copy: update it IN THIS CELL before anything imports
    # it. --ff-only means only fast-forwards happen, so a dirty/runtime
    # copy is never mangled by a merge; on failure keep the old code.
    try:
        r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"],
                           capture_output=True, text=True, timeout=120)
        print((r.stdout or r.stderr or "already up to date").strip())
    except Exception as e:
        print("git pull skipped:", e)
    # If this kernel imported src/ before the pull, drop it from the module
    # cache so the fresh code is used, not the stale in-memory copy.
    for mod in list(sys.modules):
        if mod == "src" or mod.startswith("src."):
            del sys.modules[mod]

if not repo and REPO_URL:
    if IN_COLAB:
        dest = f"/content/{REPO_NAME}"
    elif IN_KAGGLE:
        dest = f"/kaggle/working/{REPO_NAME}"
    else:
        dest = os.path.join(os.path.abspath(""), REPO_NAME)
    try:  # 1. git clone (fast, updates later with git pull)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, dest], check=True)
        repo = dest if has_repo(dest) else find_repo_local()
    except Exception as e:
        print("git clone failed:", e)
        try:  # 2. zipball download (the wget equivalent)
            print("downloading zipball...")
            req = urllib.request.Request(ZIPBALL, headers={"User-Agent": "nano-gpt-lab"})
            data = urllib.request.urlopen(req, timeout=120).read()
            os.makedirs(dest, exist_ok=True)
            repo = extract_zip(io.BytesIO(data), dest)
        except Exception as e2:
            print("zipball failed:", e2)

if not repo and IN_COLAB:   # 3. last resort: manual upload
    from google.colab import files
    print("upload nano-gpt-lab.zip:")
    files.upload()
    dest = f"/content/{REPO_NAME}"
    for zf in glob.glob(f"/content/{REPO_NAME}.zip") + glob.glob("/content/nano-gpt-lab.zip"):
        repo = extract_zip(zf, dest)

if not repo and IN_KAGGLE:  # 3. last resort: a zip added as a Dataset input
    dest = f"/kaggle/working/{REPO_NAME}"
    for zf in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        try:
            r = extract_zip(zf, dest)
            if r:
                repo = r
                break
        except Exception:
            continue
    for d in glob.glob(f"/kaggle/input/**/{REPO_NAME}", recursive=True):
        if has_repo(d):
            repo = d
            break

assert repo, ("could not fetch the repo - check REPO_URL / your network, "
              "or upload the zip manually")
print("repo at:", repo)
os.chdir(repo)
sys.path.insert(0, repo)

In [ ]:
# --- 2. DEPS + ENVIRONMENT ------------------------------------------------
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyyaml", "tqdm", "matplotlib", "requests"],
               capture_output=True)
from src.utils.environment import print_environment
print_environment()

In [ ]:
# --- 3. SMOKE TESTS (always cheap, always worth it) ------------------------
%run scripts/run_tests.py

In [ ]:
# --- 4. SETTINGS - edit these lines ----------------------------------------
DATASET = "shakespeare"        # shakespeare | tinystories | openwebtext
SIZE    = "gpt2"               # nano (local) | small | gpt2 | gpt3
STEPS   = 2000                 # training steps per variant
SEEDS   = "42"                 # "42 123 2026" for the full seed set
DEVICES = "auto"               # auto (all GPUs) | 1 | 2 (Kaggle T4x2)

# --- RUN THE WHOLE A/B/C/D COMPARISON (identical recipe, one variable) -----
%run scripts/train.py --dataset {DATASET} --compare --size {SIZE} \
    --steps {STEPS} --seeds {SEEDS} --devices {DEVICES}

In [ ]:
# --- 5. THE LAYMAN REPORTS -------------------------------------------------
%run scripts/analyze.py --dataset {DATASET} --size {SIZE}

from IPython.display import Image, display
import glob
from src.utils.environment import repo_root
for png in sorted(glob.glob(os.path.join(repo_root(), "results", DATASET, "*.png"))):
    print(os.path.basename(png))
    display(Image(png))

**Where your work went:**

- Colab: `/content/drive/MyDrive/nano-gpt-lab/runs/...` (checkpoints mirrored to Drive every checkpoint interval + at the end).
- Kaggle: `/kaggle/working/nano-gpt-lab/runs/...` - persisted to your Kaggle account automatically as the notebook output.
- Local: `runs/` inside the repo.

Resume after a disconnect: same command with `--resume`.
TinyStories next: set DATASET above to `tinystories` (downloads a 50MB documented subset).